# PyBaMM Electrode Diffusion Sensitivity Analysis
**Last updated: 2026-04-03 13:29 (Local Time)**

This notebook sweeps anode vs cathode solid-state diffusion coefficients independently,
with a configurable SEI mechanism. Run ONE VARIANT AT A TIME, then **restart the runtime**.

Diffusion variants:
* `"Anode 0.1x, Cathode 1x"`
* `"Anode 10x, Cathode 1x"`
* `"Anode 1x, Cathode 0.1x"`
* `"Anode 1x, Cathode 10x"`

SEI models:
* `"reaction limited"`
* `"solvent-diffusion limited"`
* `"electron-migration limited"`
* `"interstitial-diffusion limited"`

In [ ]:
!pip install pybamm==26.3 pandas matplotlib

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

pybamm.set_logging_level("NOTICE")

## Configuration
**Change BOTH variables below, then Run All. Restart runtime between runs.**

In [ ]:
CURRENT_VARIANT = "Anode 0.1x, Cathode 1x"  # <--- Change this!
CURRENT_SEI = "solvent-diffusion limited"     # <--- Change this!

# Variant definitions: (anode_scale, cathode_scale)
VARIANT_MAP = {
    "Anode 0.1x, Cathode 1x":  (0.1, 1.0),
    "Anode 10x, Cathode 1x":   (10.0, 1.0),
    "Anode 1x, Cathode 0.1x":  (1.0, 0.1),
    "Anode 1x, Cathode 10x":   (1.0, 10.0),
}

STORAGE_DAYS = 30
INITIAL_SOC = 0.9
TOTAL_CYCLES = 100
CHUNK_SIZE = 10
MIN_CAPACITY = 2.0  # A.h — stop if total charge capacity drops below this

EXPERIMENT_STEP = (
    "Discharge at C/9 until 3.2 V",
    "Rest for 15 minutes",
    "Charge at C/2 until 4.1 V",
    "Hold at 4.1 V until C/20",
    "Rest for 15 minutes",
)

def get_submesh_types():
    model = pybamm.lithium_ion.DFN()
    return model.default_submesh_types.copy()

var_pts = {"x_n": 30, "x_s": 30, "x_p": 30, "r_n": 50, "r_p": 50}

## Diffusion Scaling Setup

In [ ]:
anode_scale, cathode_scale = VARIANT_MAP[CURRENT_VARIANT]
RUN_LABEL = f"{CURRENT_VARIANT} | SEI: {CURRENT_SEI}"

def get_scaled_parameters(anode_sf, cathode_sf):
    pv = pybamm.ParameterValues("OKane2022")

    if anode_sf != 1.0:
        base_Dn = pv["Negative particle diffusivity [m2.s-1]"]
        def Dn_scaled(sto, T, sf=anode_sf, base_fn=base_Dn):
            return sf * base_fn(sto, T)
        pv["Negative particle diffusivity [m2.s-1]"] = Dn_scaled

    if cathode_sf != 1.0:
        base_Dp = pv["Positive particle diffusivity [m2.s-1]"]
        def Dp_scaled(sto, T, sf=cathode_sf, base_fn=base_Dp):
            return sf * base_fn(sto, T)
        pv["Positive particle diffusivity [m2.s-1]"] = Dp_scaled

    return pv

print(f"Run: {RUN_LABEL}")
print(f"  Anode diffusivity scale:   {anode_scale}x")
print(f"  Cathode diffusivity scale: {cathode_scale}x")
print(f"  SEI mechanism:             {CURRENT_SEI}")

## Execution

In [ ]:
print(f"\n{'='*60}")
print(f"Running: {RUN_LABEL}")
print(f"{'='*60}\n")

options = {
    "SEI": CURRENT_SEI,
    "SEI porosity change": "true",
    "lithium plating": "partially reversible",
    "lithium plating porosity change": "true",
    "particle mechanics": ("swelling and cracking", "swelling only"),
    "SEI on cracks": "true",
    "loss of active material": "stress-driven",
}

solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-6)

# --- Ageing Phase ---
print(f"-> Storage Phase ({STORAGE_DAYS} days)")
model_age = pybamm.lithium_ion.DFN(options)
pv_age = get_scaled_parameters(anode_scale, cathode_scale)
pv_age["Current function [A]"] = 0

seconds = STORAGE_DAYS * 24 * 60 * 60
t_eval = np.linspace(0, seconds, min(100, max(20, STORAGE_DAYS * 2)))
sim_age = pybamm.Simulation(
    model_age, parameter_values=pv_age, solver=solver,
    var_pts=var_pts, submesh_types=get_submesh_types()
)
sol_ageing = sim_age.solve(t_eval=t_eval, initial_soc=INITIAL_SOC)
print("   Storage complete.")

# --- Cycling Phase ---
print(f"-> Cycling Phase ({TOTAL_CYCLES} cycles, chunked by {CHUNK_SIZE})")
print(f"   Early stop threshold: total capacity < {MIN_CAPACITY} A.h")
num_chunks = TOTAL_CYCLES // CHUNK_SIZE
experiment_chunk = pybamm.Experiment([EXPERIMENT_STEP] * CHUNK_SIZE)

cc_caps, cv_caps, dis_caps, cycles = [], [], [], []
current_solution = sol_ageing
pv_cyc = get_scaled_parameters(anode_scale, cathode_scale)
capacity_exhausted = False

for chunk_idx in range(num_chunks):
    if capacity_exhausted:
        break

    start_cycle = chunk_idx * CHUNK_SIZE + 1
    print(f"   Chunk {chunk_idx + 1}/{num_chunks} (Cycles {start_cycle}-{start_cycle + CHUNK_SIZE - 1})... ", end="")

    model_cyc = pybamm.lithium_ion.DFN(options)

    sim_cyc = pybamm.Simulation(
        model_cyc,
        experiment=experiment_chunk,
        parameter_values=pv_cyc,
        solver=solver,
        var_pts=var_pts,
        submesh_types=get_submesh_types(),
    )

    sim_cyc.model.set_initial_conditions_from(current_solution)

    try:
        sim_cyc.solve()
        sol = sim_cyc.solution

        for i, cycle_sol in enumerate(sol.cycles):
            current_cycle_num = start_cycle + i
            steps = cycle_sol.steps

            step_dis = steps[0] if len(steps) > 0 else None
            step_cc  = steps[2] if len(steps) > 2 else None
            step_cv  = steps[3] if len(steps) > 3 else None

            d_cap = abs(
                step_dis["Discharge capacity [A.h]"].entries[-1]
                - step_dis["Discharge capacity [A.h]"].entries[0]
            ) if step_dis else 0.0

            c_cap = abs(
                step_cc["Discharge capacity [A.h]"].entries[-1]
                - step_cc["Discharge capacity [A.h]"].entries[0]
            ) if step_cc else 0.0

            v_cap = abs(
                step_cv["Discharge capacity [A.h]"].entries[-1]
                - step_cv["Discharge capacity [A.h]"].entries[0]
            ) if step_cv else 0.0

            total_cap = c_cap + v_cap

            if total_cap < MIN_CAPACITY:
                print(f"\n   *** Capacity dropped to {total_cap:.3f} A.h at cycle {current_cycle_num}. Stopping. ***")
                capacity_exhausted = True
                break

            dis_caps.append(d_cap)
            cc_caps.append(c_cap)
            cv_caps.append(v_cap)
            cycles.append(current_cycle_num)

        current_solution = sim_cyc.solution
        if not capacity_exhausted:
            print("Done.")

    except Exception as e:
        print(f"STOPPED - {e}")
        break

    del sim_cyc
    gc.collect()

print(f"\nCompleted {len(cycles)} cycles total.")

## Analysis and Data Export

In [ ]:
rows = []
for c, cc, cv in zip(cycles, cc_caps, cv_caps):
    rows.append({
        "Variant": CURRENT_VARIANT,
        "SEI Model": CURRENT_SEI,
        "Cycle": c,
        "CC Capacity [A.h]": cc,
        "CV Capacity [A.h]": cv,
        "CC/CV Ratio": cc / cv if cv > 0 else float('nan')
    })

df = pd.DataFrame(rows)

milestones = [10, 50, 100, 150, 200]
df_milestones = df[df['Cycle'].isin(milestones)]
print(f"\n--- {RUN_LABEL} Milestone Table ---")
print(df_milestones.to_string(index=False))

safe_variant = CURRENT_VARIANT.replace(' ', '_').replace(',', '').replace('/', '_')
safe_sei = CURRENT_SEI.replace(' ', '_').replace('-', '_')
filename = f"diffusion_{safe_variant}_{safe_sei}_data.csv"
df.to_csv(filename, index=False)
print(f"\nSaved {len(cycles)} cycles to: {filename}")
print("Download this CSV before restarting the runtime!")

## Visual Plots

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

axs[0].plot(cycles, cc_caps, marker=".", label=CURRENT_VARIANT, color='blue')
axs[0].set_title("CC Capacity Decay")
axs[0].set_ylabel("Capacity [A.h]")
axs[0].set_xlabel("Cycles")
axs[0].grid(True)
axs[0].legend()

axs[1].plot(cycles, cv_caps, marker=".", label=CURRENT_VARIANT, color='orange')
axs[1].set_title("CV Capacity Reliance")
axs[1].set_ylabel("Capacity [A.h]")
axs[1].set_xlabel("Cycles")
axs[1].grid(True)
axs[1].legend()

ratio = [cc / cv if cv > 0 else 0 for cc, cv in zip(cc_caps, cv_caps)]
axs[2].plot(cycles, ratio, marker=".", label=CURRENT_VARIANT, color='green')
axs[2].set_title("CC/CV Ratio")
axs[2].set_ylabel("Ratio")
axs[2].set_xlabel("Cycles")
axs[2].grid(True)
axs[2].legend()

plt.suptitle(f"{RUN_LABEL}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()